# 01 — Kaggle setup + data

**Before running anything, set these in the right sidebar → Settings:**

| Setting | Value | Why |
|---|---|---|
| Accelerator | **GPU T4 x2** | 2×15 GB; FastSpeech2 fits in one |
| Internet | **On** | git clone, pip, HuggingFace |
| Persistence | **Files only** | `/kaggle/working` survives a commit |
| Environment | Pin to a stable image | torch/torchaudio versions matter |

Rules that bite people:

* GPU sessions stop at **~9 h** and `/kaggle/temp` is wiped. Everything you want to keep must be under `/kaggle/working` **and** the notebook must be *Save Version → Save & Run All (Commit)*.
* Only 20 GB of `/kaggle/working` is kept. 5 h of 22 kHz wav ≈ 1.5 GB — fine.
* One session = one notebook. Training that needs 2 sessions must be split by `--resume`.

**Upload the code once as a Kaggle Dataset** (`sstf-code`) or clone it from GitHub, then `cp -r` it into `/kaggle/working` so the repo is writable.

In [ ]:
LANG = "Telugu"  # English run complete -> Telugu is now the active language
SSTF = "/kaggle/working/sstf"

import glob, os, shutil, sys
from pathlib import Path

INPUT = Path("/kaggle/input")


def find_project_root():
    """Deepest-agnostic: any mounted dir holding common/config.py + project/."""
    hits = []
    for cfg in INPUT.rglob("common/config.py"):
        root = cfg.parent.parent
        if (root / "project").is_dir() and (root / "notebooks").is_dir():
            hits.append((len(root.parts), root))
    return sorted(hits)[0][1] if hits else None


src = find_project_root()

if src is None:
    print("No project code found. Shallow view of what IS mounted:")
    shown = 0
    for p in sorted(INPUT.rglob("*")):
        rel = p.relative_to(INPUT)
        if len(rel.parts) <= 3:
            print("  " * (len(rel.parts) - 1) + p.name + ("/" if p.is_dir() else ""))
            shown += 1
            if shown > 60:
                print("  ...")
                break
    raise SystemExit(
        "Attach the code dataset: Add Input -> Your Datasets -> sstf-code. "
        "If it is listed but not mounted, toggle it off/on and wait for 'Ready'."
    )

if os.path.exists(SSTF):
    shutil.rmtree(SSTF)
shutil.copytree(src, SSTF)
os.chdir(SSTF)
sys.path.insert(0, SSTF)
print("code source :", src)
print("working copy:", SSTF)
print(sorted(os.listdir(SSTF)))

In [ ]:
# 2) system + python deps
!sudo apt-get -qq update && sudo apt-get -qq install -y espeak-ng
!pip install -q -r requirements.txt
import torch, torchaudio
print(torch.__version__, torch.cuda.is_available(), torch.cuda.device_count())

In [ ]:
# 3) upstream FastSpeech2 clone (paper reference implementation).
# The harness expects two per-language copies:
#   third_party/FastSpeech2_English   (kept pristine)
#   third_party/FastSpeech2_Telugu    (patched by common/patch_repo.py later)
# Clone once, then duplicate to the Telugu name.
import os, shutil
from pathlib import Path
base = Path("/kaggle/working/sstf/third_party")
base.mkdir(parents=True, exist_ok=True)

en = base / "FastSpeech2_English"
if not (en / "train.py").exists():
    !git clone --depth 1 https://github.com/ming024/FastSpeech2 {en}

te = base / "FastSpeech2_Telugu"
if not (te / "train.py").exists():
    shutil.copytree(en, te)

print("English repo ok:", (en / "train.py").exists(), "->", en)
print("Telugu repo ok:", (te / "train.py").exists(), "->", te)
print(os.listdir(en)[:12])


In [ ]:
# 4) sanity: repo present, deps importable, GPU visible
import importlib, os, sys
sys.path.insert(0, "/kaggle/working/sstf")
from common import config
print("repo:", config.require_repo(LANG))
import torch, torchaudio, num2words, soundfile, tgt
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "| devices", torch.cuda.device_count())
print("torchaudio", torchaudio.__version__, "| mms", torchaudio.pipelines.MMS_FA.sample_rate)

In [ ]:
# 5) what datasets did you attach?  (Kaggle → Add Data → your upload)
import glob
for p in sorted(glob.glob("/kaggle/input/*")):
    print(p)
    for f in sorted(glob.glob(p + "/*"))[:8]:
        print("   ", f)

### 5b — Svarah (English) → wav + metadata.csv

Svarah is Indian-accented English on HuggingFace. **Check the column names first**
(`ds.features`) — the export below assumes `audio`, `text`, `speaker`.

In [ ]:
if LANG == "English":  # skip silently on a Telugu run
    # Svarah is GATED (auto-approve). Accept the terms on the dataset page once,
    # then store a read token as a Kaggle Secret named HF_TOKEN
    # (Add-ons -> Secrets). Without it this cell cannot download.
    import os, pandas as pd, soundfile as sf
    import re
    from pathlib import Path
    from datasets import load_dataset, Audio

    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        HF_TOKEN = os.environ.get("HF_TOKEN", "")   # or paste the token string here

    # the card advertises a single split called "test"
    ds = load_dataset("ai4bharat/Svarah", split="test", token=(HF_TOKEN.strip() or None))
    print(ds)
    print("columns:", ds.column_names)

    # Cast to real string paths first (decode=False): otherwise audio_filepath yields
    # an AudioDecoder object instead of a filename.
    ds_paths = ds.cast_column("audio_filepath", Audio(decode=False))
    print("example path:", ds_paths[0]["audio_filepath"]["path"])

    # The audio_filepath in Svarah is just a filename (no directory structure).
    # Use native_place_district as speaker proxy (65 districts vs 117 real speakers),
    # falling back to an index bucket when the value is missing/unknown ("unknown",
    # NaN, etc.). Documented as a limitation in the report.
    def norm_spk(name, idx):
        s = re.sub(r"[^A-Za-z0-9]+", "_", str(name)).strip("_").lower()
        if not s or s in {"unknown", "nan", "none", "n/a"}:
            return f"spk_{idx % 65:02d}"
        return s

    speakers = pd.Series([norm_spk(rec["native_place_district"], i)
                          for i, rec in enumerate(ds_paths)])
    print("speakers:", speakers.nunique(), "| total hours:", round(sum(ds["duration"]) / 3600, 2))
    print(speakers.value_counts().head(10))


In [ ]:
if LANG == "English":  # skip silently on a Telugu run
    # Speaker proxy: Svarah's paths contain no speaker id, so use native_place_district
    # (with index fallback when unknown) - documented as a limitation in the report.
    import io, os
    from pathlib import Path
    import librosa, pandas as pd, soundfile as sf
    from tqdm.auto import tqdm

    OUT = "/kaggle/working/data/English/audio"
    os.makedirs(OUT, exist_ok=True)
    TARGET_SR = 22050

    # audio_filepath entries are bare filenames (no directory structure); locate the
    # directory that actually holds the downloaded wav files once.
    def _audio_root():
        probe = ds_paths[0]["audio_filepath"]["path"]
        base = Path(ds_paths.cache_files[0]["filename"]).parent
        for parent in [base, *list(base.parents)[:5]]:
            if (parent / probe).exists():
                return parent
        for root_c in ("/kaggle/input", "/root/.cache/huggingface/datasets"):
            root_p = Path(root_c)
            if root_p.exists():
                hits = list(root_p.rglob(probe))[:1]
                if hits:
                    return hits[0].parent
        return Path()

    AROOT = _audio_root()

    dsm = ds_paths
    rows = []
    for i, rec in enumerate(tqdm(dsm)):
        a = rec["audio_filepath"]
        spk = norm_spk(rec["native_place_district"], i)

        path = a["path"]
        if not os.path.isabs(path) or not os.path.exists(path):
            path = str(AROOT / os.path.basename(path))

        if a.get("bytes"):
            y, sr = sf.read(io.BytesIO(a["bytes"]))
        else:
            y, sr = sf.read(path)
        if y.ndim > 1:
            y = y.mean(axis=1)
        if sr != TARGET_SR:
            y = librosa.resample(y.astype("float32"), orig_sr=sr, target_sr=TARGET_SR)
            sr = TARGET_SR

        name = f"{spk}_{i:06d}.wav"
        sf.write(os.path.join(OUT, name), y, sr)
        rows.append({"file": name, "text": rec["text"], "speaker": spk})

    df = pd.DataFrame(rows)
    df.to_csv("/kaggle/working/data/English/metadata.csv", index=False, encoding="utf-8")
    print(len(df), "utts |", df.speaker.nunique(), "speakers |",
          round(len(df) / df.speaker.value_counts().median(), 1), "utts/median-speaker")
    print(df.speaker.value_counts().head(10))


### 5c — IndicTTS (Telugu)

Request access at IITM DonLab. If it has not arrived, **do not stall**: the
AI4Bharat Kathbath Telugu shard is a valid stand-in — say so in the report.

Either way you end up with the same two things: `data/Telugu/audio/*.wav` and
`data/Telugu/metadata.csv` with columns `file,text,speaker`.

In [ ]:
# 5c — IndicTTS Telugu download
# This dataset is on Hugging Face (SPRINGLab/IndicTTS_Telugu)
# It's ~6.4 GB, 8576 utterances, 8.7 hours, 2 speakers (male/female)

import io, os
from pathlib import Path
from datasets import load_dataset, Audio
import soundfile as sf
from tqdm.auto import tqdm

HF_TOKEN = os.environ.get("HF_TOKEN", "")

print("Loading IndicTTS Telugu...")
ds_te = load_dataset("SPRINGLab/IndicTTS_Telugu", split="train", token=(HF_TOKEN.strip() or None))
print(ds_te)
print("columns:", ds_te.column_names)

# Cast the audio column to string paths (decode=False) so it returns a dict
# {path, bytes} instead of an AudioDecoder object.
ds_te = ds_te.cast_column("audio", Audio(decode=False))
print("example path:", ds_te[0]["audio"]["path"])

def _telugu_audio_root():
    probe = ds_te[0]["audio"]["path"]
    base = Path(ds_te.cache_files[0]["filename"]).parent
    for parent in [base, *list(base.parents)[:5]]:
        if (parent / probe).exists():
            return parent
    for root_c in ("/kaggle/input", "/root/.cache/huggingface/datasets"):
        root_p = Path(root_c)
        if root_p.exists():
            hits = list(root_p.rglob(probe))[:1]
            if hits:
                return hits[0].parent
    return Path()

AROOT = _telugu_audio_root()

OUT = "/kaggle/working/data/Telugu/audio"
os.makedirs(OUT, exist_ok=True)
TARGET_SR = 22050

import librosa
rows = []
for i, rec in enumerate(tqdm(ds_te)):
    a = rec["audio"]
    path = a["path"]
    if not os.path.isabs(path) or not os.path.exists(path):
        path = str(AROOT / os.path.basename(path))

    if a.get("bytes"):
        y, sr = sf.read(io.BytesIO(a["bytes"]))
    else:
        y, sr = sf.read(path)
    if y.ndim > 1:
        y = y.mean(axis=1)
    if sr != TARGET_SR:
        y = librosa.resample(y.astype("float32"), orig_sr=sr, target_sr=TARGET_SR)
        sr = TARGET_SR

    name = f"tel_{i:06d}.wav"
    sf.write(os.path.join(OUT, name), y, TARGET_SR)
    rows.append({"file": name, "text": rec["text"],
                 "speaker": rec.get("speaker", rec.get("gender", "tel_spk"))})

import pandas as pd
df = pd.DataFrame(rows)
df.to_csv("/kaggle/working/data/Telugu/metadata.csv", index=False, encoding="utf-8")
print(len(df), "utts |", df.speaker.nunique(), "speakers")
print(df.speaker.value_counts().head(10))


In [ ]:
import pandas as pd, soundfile as sf, glob, os

# If your upload is <wav> + <transcript.txt> one-line-per-utt, pair them here.
AUDIO = "/kaggle/working/data/Telugu/audio"
rows = []
for wav in sorted(glob.glob(AUDIO + "/*.wav")):
    rows.append({"file": os.path.basename(wav), "text": "", "speaker": "tel_spk1"})
print(len(rows), "wav files found - fill the text column from the dataset transcripts")
# pd.DataFrame(rows).to_csv("/kaggle/working/data/Telugu/metadata.csv", index=False)

In [ ]:
# 6) point the project configs at these paths, then build the 5h/1h split
import yaml, os
from pathlib import Path
p = f"{SSTF}/project/{LANG}/config.yaml"
cfg = yaml.safe_load(open(p, encoding="utf-8"))

# The export cells write to /kaggle/working/data/<LANG>; link it into the
# project tree so the manifest stage (and later notebooks) can see it.
os.makedirs(f"{SSTF}/data", exist_ok=True)
if not os.path.exists(f"{SSTF}/data/{LANG}"):
    os.symlink(f"/kaggle/working/data/{LANG}", f"{SSTF}/data/{LANG}")

data_dir = Path(f"{SSTF}/data/{LANG}")
if not (data_dir / "metadata.csv").exists():
    raise SystemExit(
        f"{data_dir}/metadata.csv not found - export the {LANG} dataset first "
        "(run the export cell for this language)"
    )
!python project/{LANG}/preprocess.py --stage manifest


**Check `project/<lang>/data/split_stats.json`**: `train_hours ≈ 5`, `val_hours ≈ 1`,
`overlap: 0`. If hours are short, the audio dir or the metadata paths are wrong —
fix that now, not after training.

In [ ]:
# 7) free disk: the Hugging Face parquet cache is ~6.4 GB for Telugu and is no
#    longer needed once metadata.csv + audio/ exist.  /kaggle/working has a
#    ~20 GB budget and 02_preprocess writes the mel/pitch/energy features next.
import shutil, subprocess
from pathlib import Path

for cache in (Path.home() / ".cache/huggingface/datasets",
              Path("/root/.cache/huggingface/datasets")):
    if cache.exists():
        shutil.rmtree(cache, ignore_errors=True)
        print("removed", cache)

DATA = Path(f"/kaggle/working/data/{LANG}")
csv_p = DATA / "metadata.csv"
aud = DATA / "audio"
n = len(list(aud.glob("*.wav"))) if aud.is_dir() else 0
print(f"{LANG}: {n} wavs | metadata.csv exists: {csv_p.exists()}")
assert csv_p.exists() and n > 0, f"dataset export for {LANG} did not complete"

subprocess.run(["df", "-h", "/kaggle/working"])
